# broadcasting-rules composite — cx28: repeat a row across batch axis then broadcast-add

> Composite procedural drill from [Delta Drills](https://delta-drills.vercel.app).
> Exercises 2 atoms together: `broadcasting-rules`, `einops-repeat`
> Running the final beacon reports progress against all 2 subtopics.

**Why composite drills.** Single-atom drills test atomic skills in isolation. Composite drills test the COMPOSITION — how atoms wire together in real ARENA code. Passing this drill demonstrates you can apply the atoms jointly, not just individually.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)

## Connect to Delta Drills

Paste your Delta Drills auth token below. Beacon will report progress against ALL atoms exercised by this composite.

In [ ]:
# === Delta Drills auth (composite) ===
DD_TOKEN = ""  # paste token, then run
DD_PRIMARY_ATOM = "broadcasting-rules"
DD_ATOM_IDS = ["broadcasting-rules", "einops-repeat"]
DD_SUBTOPICS = ["Numpy: Vectorization and broadcasting", "Einops: Repeat"]
DD_BACKEND_URL = "https://delta-drills-backend.fly.dev"

_dd_passed = set()

## How these two atoms compose

Per-sample bias addition is the canonical broadcasting use case: you have a `(D,)` bias vector and an `(B, D)` activation matrix; you want each row of activations shifted by the bias. Naïve broadcasting `acts + bias` works (it auto-prepends a 1-axis), but it's implicit.

`einops.repeat` makes the broadcast EXPLICIT — `repeat(bias, 'd -> b d', b=B)` materializes the broadcast intent in the code. The trick: einops compiles this to `expand` (stride-0 view, no copy), so the result is identical in storage and value to letting broadcasting handle it — but the named-axis pattern documents WHICH axis the bias broadcasts across.

### Composite Exercise — repeat a row across batch axis then broadcast-add

**Atoms exercised together**: `broadcasting-rules`, `einops-repeat`

Implement `cx28_add_bias_via_repeat(acts, bias)` that adds a per-feature bias to a batch of activations.

- `acts` has shape `(B, D)` — the batch of activations.
- `bias` has shape `(D,)` — the per-feature bias vector.

1. **Repeat** the bias across the batch axis: `repeat(bias, 'd -> b d', b=B)` so it has shape `(B, D)`.
2. **Broadcast-add** the repeated bias to the activations. (At this point the shapes match so it's just `acts + bias_b`, but the test verifies the named-repeat path matches the implicit-broadcasting path bit-for-bit.)

Return shape `(B, D)`. Cross-check against the plain `acts + bias` (implicit broadcast).

In [ ]:
# Fill in the function below, then run this cell. The test asserts the composition is correct.

def cx28_add_bias_via_repeat(acts, bias):
    raise NotImplementedError

def _test_cx28():
    # Case A: cross-check against implicit broadcasting.
    acts = t.randn(4, 8)
    bias = t.randn(8)
    out = cx28_add_bias_via_repeat(acts, bias)
    assert tuple(out.shape) == (4, 8), f'shape: {tuple(out.shape)}'
    assert t.allclose(out, acts + bias), 'repeat+add must equal implicit-broadcast result'

    # Case B: hand-check on distinguishable values.
    acts2 = t.zeros(3, 4)
    bias2 = t.tensor([1.0, 2.0, 3.0, 4.0])
    out2 = cx28_add_bias_via_repeat(acts2, bias2)
    # Every row should equal the bias.
    for r in range(3):
        assert t.equal(out2[r], bias2), f'row {r} mismatch: {out2[r]}'

    # Case C: realistic linear-layer scale.
    acts3 = t.randn(32, 768)
    bias3 = t.randn(768)
    out3 = cx28_add_bias_via_repeat(acts3, bias3)
    assert tuple(out3.shape) == (32, 768)
    assert t.allclose(out3, acts3 + bias3)

    # Case D: batch of 1 — degenerate but must still work.
    acts4 = t.randn(1, 5)
    bias4 = t.randn(5)
    out4 = cx28_add_bias_via_repeat(acts4, bias4)
    assert tuple(out4.shape) == (1, 5)
    assert t.allclose(out4, acts4 + bias4)
    # --- atom-coverage: einops.repeat must be used AND yield a stride-0 view; plain `+` is not allowed alone ---
    import inspect as _inspect
    _src = _inspect.getsource(cx28_add_bias_via_repeat)
    assert 'repeat(' in _src, 'must use einops.repeat to build the (B,D) bias tensor'
    assert '.expand(' not in _src and 'expand_as' not in _src, 'must use einops.repeat, not torch.expand'
    assert 'broadcast_to' not in _src, 'must use einops.repeat, not broadcast_to'
    _g = cx28_add_bias_via_repeat.__globals__
    _orig_repeat = _g.get('repeat')
    _calls = []
    def _spy_repeat(*a, **kw):
        r = _orig_repeat(*a, **kw)
        _calls.append(r)
        return r
    _g['repeat'] = _spy_repeat
    try:
        cx28_add_bias_via_repeat(t.randn(4, 8), t.randn(8))
    finally:
        _g['repeat'] = _orig_repeat
    assert len(_calls) >= 1, 'cx28_add_bias_via_repeat must call einops.repeat'
    # The b-axis on the broadcast bias must be stride-0 (no copy along the batch).
    assert any(0 in r.stride() for r in _calls), (
        'einops.repeat output must be a stride-0 broadcast view along the batch axis'
    )

    _dd_passed.add('cx28')

_test_cx28()

<details><summary>Show solution — cx28</summary>

```python
def cx28_add_bias_via_repeat(acts, bias):
    B = acts.shape[0]
    # Atom A (einops-repeat): repeat the (D,) bias into (B, D) — stride-0 view, no copy.
    bias_b = repeat(bias, 'd -> b d', b=B)
    # Atom B (broadcasting-rules): shapes now match, but the SAME result drops out from
    # plain `acts + bias` because broadcasting auto-prepends a 1-axis. The named repeat
    # makes the intent explicit, while broadcasting + repeat-as-expand keep it free.
    return acts + bias_b
```

`einops.repeat(bias, 'd -> b d', b=B)` is identical in storage to `bias.expand(B, -1)` — both produce a stride-0 view of the original `bias` buffer. So the cost of the explicit-repeat path is exactly zero compared to the implicit-broadcast path, but the named pattern documents WHICH axis broadcasts across. This is the einops-as-self-documenting-broadcast idiom — useful in any pipeline where multiple shapes are converging.
</details>

## Report completion

Run the cell below to send progress to Delta Drills. The beacon fires once and reports all 2 subtopics together.

In [ ]:
# === Delta Drills completion beacon (composite — fires for ALL atoms) ===
import urllib.request as _dd_req, json as _dd_json

_DD_REQUIRED = {'cx28'}

def report_completion():
    missing = _DD_REQUIRED - _dd_passed
    if missing:
        print(f"[Delta Drills] {sorted(missing)} not yet passing — fix the cell above, then re-run this one.")
        return
    if not DD_TOKEN:
        print('[Delta Drills] DD_TOKEN is empty — completion not reported.')
        return
    body = _dd_json.dumps({
        'exercise_title': f'composite-drill:{DD_PRIMARY_ATOM}:cx28',
        'subtopics': ["Numpy: Vectorization and broadcasting", "Einops: Repeat"],
        'feedback': 'somewhat',
        'correct': True,
    }).encode('utf-8')
    req = _dd_req.Request(
        f'{DD_BACKEND_URL}/api/practice/arena-rating',
        data=body,
        headers={
            'Content-Type': 'application/json',
            'Authorization': f'Bearer {DD_TOKEN}',
        },
        method='POST',
    )
    try:
        with _dd_req.urlopen(req, timeout=5) as r:
            resp = _dd_json.loads(r.read())
        print(f'[Delta Drills] reported composite (atoms={DD_ATOM_IDS})')
        print(f'[Delta Drills] EWMA updated: {resp}')
    except Exception as e:
        print(f'[Delta Drills] beacon failed: {e}')

report_completion()